# MediaPipe Pose Preprocessing Pipeline

This notebook applies, step by step, the preprocessing pipeline defined in `sources/preprocessing.py`.

**Philosophy:**

- All computation logic is in the Python module; here we **orchestrate** and **verify visually**.
- One cell = one conceptual step.
- Modifiable parameters (filter, paths) are gathered at the top of the notebook.
- We perform visual checks on a single participant while building the full dataset in the background.

## 1. Import libraries and local module

Import the `preprocessing` module from `sources/`. To make the import work, add `sources/` to `sys.path`.

The rest is the minimal scientific stack useful for visual verification.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
SOURCES_DIR = PROJECT_DIR / "sources"
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
if str(SOURCES_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCES_DIR))

import preprocessing as pp

# Auto-reload preprocessing.py whenever it changes on disk
%load_ext autoreload
%autoreload 2

#pd.set_option("display.max_columns", 50)
#pd.set_option("display.width", 200)
#print(f"Project dir  : {PROJECT_DIR}")
#print(f"Notebooks    : {NOTEBOOKS_DIR}")
#print(f"Sources      : {SOURCES_DIR}")
#print(f"Data         : {DATA_DIR}")
#print(f"Results      : {RESULTS_DIR}")
#print("Module `preprocessing` imported successfully.")

## 2. Pipeline parameters

Centralize here **all modifiable parameters** of the pipeline:

- paths (folders in `data/`),
- parameters of the low-pass Butterworth filter.

To test other filter values, just edit this cell and run the notebook again.

In [2]:
# --- Paths ---
DATA_DIR = PROJECT_DIR / "data"

# --- Filter parameters (edit here to test different values) ---
FPS = 30.0          # MediaPipe sampling rate, frames per second
CUTOFF_HZ = 6.0     # Low-pass cutoff frequency, Hz
FILTER_ORDER = 2    # Butterworth order (filtfilt effectively doubles it)

#print(f"DATA_DIR     : {DATA_DIR}")
print(f"FPS          : {FPS}")
print(f"CUTOFF_HZ    : {CUTOFF_HZ}")
print(f"FILTER_ORDER : {FILTER_ORDER}")

FPS          : 30.0
CUTOFF_HZ    : 6.0
FILTER_ORDER : 2


## 3. Auto-detect files and participant identity

List all CSV files in `data/P` and `data/SZ`, then extract participant identity (`group`, `participant`, `participant_id`).

**Visual check:** 20 files in each group, 40 total.

In [3]:
p_files, sz_files, all_files = pp.list_participant_files(DATA_DIR)
participants_df, participant_rows = pp.build_participant_table(all_files)

print(f"Found {len(p_files)} files in group P")
print(f"Found {len(sz_files)} files in group SZ")
print(f"Total             : {len(all_files)} files")
#participants_df.head(10)

Found 20 files in group P
Found 20 files in group SZ
Total             : 40 files


## 5. Build global long-format DataFrame

Build the long-format DataFrame containing **all participants**, filtered on landmarks of interest (head, shoulders, wrists, hips). Add identity columns `group`, `participant`, `participant_id`.

In [4]:
df_all_long = pp.build_long_dataframe(participant_rows)

print(f"Global long shape   : {df_all_long.shape}")
print(f"Unique participants : {df_all_long['participant_id'].nunique()}")
print(f"Landmarks kept      : {sorted(df_all_long['landmark'].unique())}")
#df_all_long.head(10)

Global long shape   : (250650, 8)
Unique participants : 40
Landmarks kept      : ['left_hip', 'left_shoulder', 'left_wrist', 'nose', 'right_hip', 'right_shoulder', 'right_wrist']


## 6. Invert Y axis

MediaPipe places `y = 0` at the top of the image. Apply `y = 1 - y` to get a biomechanical convention: larger `y` = higher. After this step, a raised wrist will have a higher `y` than the shoulder.

In [5]:
df_all_long = pp.invert_y_axis(df_all_long)

#print("y stats after inversion:")
#print(df_all_long["y"].describe())

## 7. Interpolate missing coordinates

Missing `x` and `y` values are interpolated linearly (per participant, per landmark, sorted by frame). Residual NaNs at series start/end are filled with forward-fill then backward-fill.

**Visual check:** NaN count on `x` and `y` should drop to zero (or nearly).

In [6]:
#before_interp_summary = pp.count_complete_landmark_signals(df_all_long)
#before_interp_summary

In [7]:
df_all_long = pp.interpolate_coordinates(df_all_long)

In [8]:
#after_interp_summary = pp.count_complete_landmark_signals(df_all_long)
#after_interp_summary

## 8. Convert to wide format

One row per `(participant, frame)`, one column per landmark coordinate (`nose_x`, `left_wrist_y`, etc.).

In [9]:
df_wide = pp.long_to_wide(df_all_long)

#print(f"Wide shape : {df_wide.shape}")
#print(f"Columns    : {list(df_wide.columns)}")
#df_wide.head()

In [10]:
coord_cols = [
    "nose_x", "nose_y",
    "left_shoulder_x", "left_shoulder_y",
    "right_shoulder_x", "right_shoulder_y",
    "left_wrist_x", "left_wrist_y",
    "right_wrist_x", "right_wrist_y",
    "left_hip_x", "left_hip_y",
    "right_hip_x", "right_hip_y",
]

coord_summary = pd.DataFrame([
    {
        "coordinate": col,
        "n_participants_total": df_wide["participant_id"].nunique(),
        "n_participants_usable": df_wide.groupby("participant_id")[col]
            .apply(lambda s: not s.isna().all())
            .sum()
    }
    for col in coord_cols
])

coord_summary

,coordinate,n_participants_total,n_participants_usable
0,nose_x,40,40
1,nose_y,40,40
2,left_shoulder_x,40,40
3,left_shoulder_y,40,40
4,right_shoulder_x,40,40
5,right_shoulder_y,40,40
6,left_wrist_x,40,37
7,left_wrist_y,40,37
8,right_wrist_x,40,36
9,right_wrist_y,40,36


## 9. Compute derived points and image center

Add:

- `mid_shoulders`: midpoint between shoulders,
- `mid_hips`: midpoint between hips,
- `torso_center`: midpoint between `mid_shoulders` and `mid_hips`,
- `image_center`: fixed point at `(0.5, 0.5)`.

In [11]:
df_wide = pp.add_derived_points(df_wide)

#df_wide[[
#    "participant_id", "frame",
#    "mid_shoulders_x", "mid_shoulders_y",
#    "mid_hips_x", "mid_hips_y",
#    "torso_center_x", "torso_center_y",
#    "image_center_x", "image_center_y",
#]]

## 10. Shoulder width and its median per participant

Compute shoulder width (2D Euclidean distance between shoulders) per frame, then take the median per participant. This median serves as a robust normalization factor for size and camera distance variations.

In [12]:
df_wide, median_shoulder_width = pp.compute_median_shoulder_width(df_wide)

#print("Median shoulder width per participant (first 10):")
#print(median_shoulder_width.head(10))
#print(f"\nRange: [{median_shoulder_width.min():.4f}, {median_shoulder_width.max():.4f}]")

## 11. Build relative movement signals

Build relative signals (hands relative to shoulders, torso oscillation, etc.) from **raw** coordinates. They are more informative than absolute coordinates because they remove global body position from the image.

In [13]:
df_wide = pp.build_movement_signals(df_wide)

movement_summary = pp.count_complete_movement_signals(df_wide)
movement_summary

# to inspect the movement signals for a specific participant, uncomment the lines below and set TEST_PID to a valid participant_id from df_wide
#TEST_PID = df_wide["participant_id"].iloc[0]
#df_wide.loc[
    #df_wide["participant_id"] == TEST_PID,
    #["participant_id", "frame"] + pp.MOVEMENT_SIGNALS,
#].head()

,movement,n_participants_total,n_participants_complete_100pct
0,head_horizontal,40,40
1,head_vertical,40,40
2,left_hand_horizontal,40,37
3,left_hand_vertical,40,37
4,right_hand_horizontal,40,36
5,right_hand_vertical,40,36
6,sideways,40,40
7,swaying,40,40


## 12. Normalize movement signals

Divide each movement signal by the **median shoulder width** of the corresponding participant. Explicitly create columns suffixed `_norm` to **preserve raw signals** side-by-side. The normalized version will be filtered in the next step.

In [14]:
df_wide = pp.normalize_movement_signals(df_wide, median_shoulder_width)

#df_wide.loc[
#    df_wide["participant_id"] == TEST_PID,
#    [
#        "participant_id", "frame",
#        "left_hand_horizontal", "left_hand_horizontal_norm",
#        "sideways", "sideways_norm",
#    ],
#].head()

## 13. Low-pass Butterworth filtering (zero-phase) on normalized signals

Apply a Butterworth low-pass filter via `filtfilt` (forward + backward, no phase shift) on **normalized** signals. Filtering is done **per participant** to avoid cross-subject contamination.

For each `s_norm`, create a column `s_filt`.

Parameters `FPS`, `CUTOFF_HZ`, and `FILTER_ORDER` come from the parameters cell at the top of the notebook.

In [15]:
df_wide = pp.filter_movement_signals(
    df_wide,
    fps=FPS,
    cutoff_hz=CUTOFF_HZ,
    order=FILTER_ORDER,
)

#print("Filtered signal columns created:")
#print([f"{s}_filt" for s in pp.MOVEMENT_SIGNALS])

## 14. Center filtered signals

Center each filtered signal by subtracting its **mean per participant**, so all signals oscillate around zero.

**Visual check:** plot for a participant the raw signal, filtered signal, and filtered+centered signal.

In [16]:
df_wide = pp.center_filtered_signals(df_wide)

#SIGNAL_TO_PLOT = "left_hand_vertical"
#sub = df_wide[df_wide["participant_id"] == TEST_PID].sort_values("frame")

#plt.figure(figsize=(11, 4))
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_norm"],          label="normalized (raw)", alpha=0.5)
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_filt"],          label="filtered",         linewidth=1, alpha=0.7)
#plt.plot(sub["frame"], sub[f"{SIGNAL_TO_PLOT}_filt_centered"], label="filtered+centered", linewidth=1)
#plt.axhline(0, color="k", linewidth=0.5)
#plt.title(f"Participant {TEST_PID} -- signal: {SIGNAL_TO_PLOT}")
#plt.xlabel("frame")
#plt.ylabel("value (shoulder-width units)")
#plt.legend()
#plt.tight_layout()
#plt.show()

## 15. Save global output CSV files

Write to `data/`:

1. **`all_participants_raw_movements.csv`**: wide format, all participants, identity columns + raw normalized signals (`*_norm`) + filtered signals (`*_filt`).
2. **`all_participants_preprocessed_movements.csv`**: wide format, all participants, identity columns + filtered and centered signals. Columns are **renamed without the `_filt_centered` suffix** for simpler use in downstream analysis notebooks.

In [17]:
raw_path, preproc_path = pp.save_outputs(df_wide, DATA_DIR)

#print(f"Saved raw movements          -> {raw_path}")
print(f"Saved preprocessed movements -> {preproc_path}")

Saved preprocessed movements -> c:\Users\factory\Desktop\pons.lea\data\all_participants_preprocessed_movements.csv
